In [1]:
import os
os.chdir('../../../../..')

In [2]:
import polars as pl

In [3]:
path = 'results/qm9/regression_benchmark/n_2000/raw_all_targets.csv'
df = pl.read_csv(path)

In [2]:
"""
Paired statistics for the flat-vs-curved comparison (Table 15).

The data are paired by SEED: every method is evaluated on the same 5 seeds
(42, 123, 456, 789, 1011), and the seed fixes the train/test split. So for two
methods, their scores at a common seed form a matched pair, and a paired test
removes the seed-to-seed variance. That is the correct structure here.

Provides:
  - paired t-test            -> use for "A beats B" claims (rejecting the null)
  - TOST equivalence test    -> use for "A and B are the same" claims
  - effect size (Cohen's dz) + 95% CI on the mean difference, always reported
"""

import numpy as np
import pandas as pd
from scipy import stats


def paired_compare(df, method_a, method_b, target, metric="test_r2",
                   equiv_margin=0.05):
    """Paired comparison of two methods on one target, matched by seed.

    method_a, method_b : method names as they appear in the 'method' column
    target             : 'gap', 'mu', or 'geometric_strain'
    metric             : column to compare (default 'test_r2')
    equiv_margin       : smallest effect of interest for the TOST equivalence
                         test, in the same units as `metric`. For R^2, 0.05
                         means "anything smaller than 0.05 is practically zero".
    """
    # keep only successful runs for the two methods on this target
    sub = df[(df.target == target) & (df.status == "ok") &
             (df.method.isin([method_a, method_b]))]

    # align scores by seed -> rows = seeds, cols = the two methods
    wide = sub.pivot_table(index="seed", columns="method", values=metric)
    wide = wide.dropna(subset=[method_a, method_b])  # keep only shared seeds

    a = wide[method_a].to_numpy()
    b = wide[method_b].to_numpy()
    diff = a - b
    n = len(diff)
    if n < 2:
        raise ValueError(f"Only {n} shared seed(s) for "
                         f"{method_a} vs {method_b} on {target}.")

    mean_d = diff.mean()
    sd_d = diff.std(ddof=1)
    se_d = sd_d / np.sqrt(n)

    # --- paired t-test (two-sided): H0 mean difference == 0 ---------------
    t_stat, p_two = stats.ttest_rel(a, b)

    # --- 95% CI on the mean difference -----------------------------------
    tcrit = stats.t.ppf(0.975, df=n - 1)
    ci_low, ci_high = mean_d - tcrit * se_d, mean_d + tcrit * se_d

    # --- Cohen's dz (paired effect size) ---------------------------------
    dz = mean_d / sd_d if sd_d > 0 else np.nan

    # --- TOST equivalence test against +/- equiv_margin ------------------
    # two one-sided tests; equivalence is established if BOTH reject.
    t_low = (mean_d - (-equiv_margin)) / se_d   # H0: diff <= -margin
    t_high = (mean_d - (equiv_margin)) / se_d    # H0: diff >= +margin
    p_low = stats.t.sf(t_low, df=n - 1)          # upper tail
    p_high = stats.t.cdf(t_high, df=n - 1)       # lower tail
    p_tost = max(p_low, p_high)
    equivalent = p_tost < 0.05

    return {
        "comparison": f"{method_a}  vs  {method_b}",
        "target": target,
        "metric": metric,
        "n_pairs": n,
        "mean_a": a.mean(),
        "mean_b": b.mean(),
        "mean_diff": mean_d,
        "ci95": (ci_low, ci_high),
        "cohens_dz": dz,
        "t_stat": t_stat,
        "p_paired_ttest": p_two,
        "equiv_margin": equiv_margin,
        "p_tost": p_tost,
        "equivalent_within_margin": equivalent,
    }


def report(res):
    lo, hi = res["ci95"]
    print(f"\n{res['comparison']}   [{res['target']}, {res['metric']}]")
    print(f"  n pairs (seeds)      : {res['n_pairs']}")
    print(f"  mean A / mean B      : {res['mean_a']:.4f} / {res['mean_b']:.4f}")
    print(f"  mean difference      : {res['mean_diff']:+.4f}  "
          f"(95% CI {lo:+.4f}, {hi:+.4f})")
    print(f"  effect size (dz)     : {res['cohens_dz']:+.3f}")
    print(f"  paired t-test p      : {res['p_paired_ttest']:.4f}   "
          f"(use to claim A != B)")
    print(f"  TOST p (margin {res['equiv_margin']:+.3f}) : {res['p_tost']:.4f}"
          f"   -> equivalent: {res['equivalent_within_margin']}   "
          f"(use to claim A == B)")



In [3]:
path = 'results/qm9/regression_benchmark/n_2000/raw_all_targets.csv'
df = pd.read_csv(path)

# Flat vs curved pairs from Table 15 (SOAP, under shared Laplacian kernel)
flat_vs_curved = [
    ("soap_riemann_logeuclidean_dist_laplacian",  # flat (log-Euclidean)
        "soap_riemann_affine_invariant"),            # curved (AIRM)
    ("soap_grassmann_chordal_dist_laplacian",     # flat (chordal)
        "soap_grassmann_geodesic_dist_laplacian"),   # curved (geodesic)
]

for target in ["gap", "mu", "geometric_strain"]:
    for flat, curved in flat_vs_curved:
        report(paired_compare(df, flat, curved, target, equiv_margin=0.05))


soap_riemann_logeuclidean_dist_laplacian  vs  soap_riemann_affine_invariant   [gap, test_r2]
  n pairs (seeds)      : 5
  mean A / mean B      : 0.6229 / 0.6293
  mean difference      : -0.0064  (95% CI -0.0161, +0.0034)
  effect size (dz)     : -0.807
  paired t-test p      : 0.1455   (use to claim A != B)
  TOST p (margin +0.050) : 0.0001   -> equivalent: True   (use to claim A == B)

soap_grassmann_chordal_dist_laplacian  vs  soap_grassmann_geodesic_dist_laplacian   [gap, test_r2]
  n pairs (seeds)      : 5
  mean A / mean B      : 0.5155 / 0.5033
  mean difference      : +0.0122  (95% CI +0.0001, +0.0243)
  effect size (dz)     : +1.248
  paired t-test p      : 0.0493   (use to claim A != B)
  TOST p (margin +0.050) : 0.0005   -> equivalent: True   (use to claim A == B)

soap_riemann_logeuclidean_dist_laplacian  vs  soap_riemann_affine_invariant   [mu, test_r2]
  n pairs (seeds)      : 5
  mean A / mean B      : 0.4006 / 0.4014
  mean difference      : -0.0008  (95% CI -0.0048, +0